In [ ]:
model_1 = "gpt-4o"
model_2 = "Qwen-VL-72B-Instruct"

model_1 = model_1.replace("-", "_")
model_2 = model_2.replace("-", "_")

In [ ]:
import datasets

data = datasets.load_dataset("lmms-lab/LiveBenchDetailedResults", "2024-09")

In [ ]:
model_data_1 = data[model_1].to_pandas()
model_data_2 = data[model_2].to_pandas()

In [ ]:
model_data_1.columns

In [ ]:
same_data = ["images", "question", "ground_truth", "criteria", "subtask", "website"]
model_data_2.drop(columns=same_data, inplace=True)
merged_data = model_data_1.merge(model_data_2, on="id", suffixes=(f"_1", f"_2"))

In [ ]:
merged_data.head()

In [ ]:
delta = merged_data["score_1"] - merged_data["score_2"]

In [ ]:
merged_data["delta"] = delta

In [ ]:
sorted_data = merged_data.sort_values("delta", ascending=False)

In [ ]:
sorted_data.columns

In [ ]:
features = data[model_1].features.copy()

In [ ]:
for feature in ["score", "reason", "response"]:
    feat = features.pop(feature)
    features[f"{feature}_1"] = feat
    features[f"{feature}_2"] = feat

In [ ]:
features["delta"] = datasets.Value("float32")

In [ ]:
features

In [ ]:
def gen():
    for i, row in sorted_data.iterrows():
        yield row.to_dict()


final_data = datasets.Dataset.from_generator(gen, features=features)

In [ ]:
final_data.push_to_hub("lmms-lab/LiveBenchDetailedResultsComparison", "2024-09", split=f"{model_1}_vs_{model_2}")